In [ ]:
# 1. Install dependencies — runtime will restart automatically after this
!pip install -q transformers peft huggingface_hub sentencepiece
!apt-get install -y -q cmake build-essential
import os; os.kill(os.getpid(), 9)

In [ ]:
# 2. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clone and build llama.cpp
!git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
!cd /content/llama.cpp && cmake -B build -DLLAMA_CURL=OFF && cmake --build build --config Release -j$(nproc) --target llama-quantize
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

In [ ]:
# 4. Merge LoRA checkpoint into base model — saves to local /content/ (not Drive)
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

HF_TOKEN   = ''  # paste your HF write token
BASE       = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
CHECKPOINT = '/content/drive/MyDrive/Lumen/lumen-checkpoints/checkpoint-2643'
MERGED     = '/content/lumen-merged'

print('Loading base model...')
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='cpu', token=HF_TOKEN)
print('Merging LoRA...')
merged = PeftModel.from_pretrained(base, CHECKPOINT)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED)
tokenizer = AutoTokenizer.from_pretrained(BASE, token=HF_TOKEN)
tokenizer.save_pretrained(MERGED)
print('Merged model saved to', MERGED)

In [ ]:
# 5. Convert to GGUF F16
import os, json

MERGED  = '/content/lumen-merged'
F16     = '/content/lumen-f16.gguf'

cfg_path = MERGED + '/tokenizer_config.json'
with open(cfg_path) as f:
    cfg = json.load(f)
cfg['tokenizer_class'] = 'PreTrainedTokenizerFast'
with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)
print('Tokenizer patched.')

!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED} --outfile {F16} --outtype f16

if os.path.exists(F16) and os.path.getsize(F16) > 1e9:
    print('F16 done. Size:', round(os.path.getsize(F16)/1e9, 1), 'GB')
else:
    print('ERROR: F16 file missing or too small.')

In [ ]:
# 6. Quantize to Q4_K_M and save to Drive (~4.5 GB)
import os

F16  = '/content/lumen-f16.gguf'
Q4   = '/content/drive/MyDrive/Lumen/lumen-q4_k_m.gguf'

!/content/llama.cpp/build/bin/llama-quantize {F16} {Q4} Q4_K_M

if os.path.exists(Q4) and os.path.getsize(Q4) > 1e9:
    print('Quantized. Size:', round(os.path.getsize(Q4)/1e9, 1), 'GB')
else:
    print('ERROR: Q4 file missing or too small.')

In [ ]:
# 7. Upload GGUF to HuggingFace
from huggingface_hub import HfApi

HF_TOKEN = ''  # paste your HF write token
Q4       = '/content/drive/MyDrive/Lumen/lumen-q4_k_m.gguf'

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=Q4,
    path_in_repo='lumen-q4_k_m.gguf',
    repo_id='RavikxxBGamin/Lumen',
    repo_type='model',
)
print('Uploaded to HuggingFace!')